In [ ]:
import sys
dice_x_path  = "/Users/volk/Documents/bau24-25/thesis/repos/DiCE-X"

# insert at position 0 so they shadow anything of the same name in
# site-packages; if you’d rather keep PyPI packages as the default, use
# sys.path.append(…) instead.

if p not in sys.path:
    sys.path.insert(0, dice_x_path)

In [ ]:
import dice_ml
from dice_ml.utils import helpers, neuralnetworks
import dice_ml_x
from dice_ml_x.utils import helpers, neuralnetworks
import pickle
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

import itertools
from random import randrange
import random
import os
from collections import OrderedDict
from scipy.spatial.distance import pdist, squareform
from sklearn.neighbors import KNeighborsClassifier
from typing import Optional, Union

import numpy as np, pandas as pd, random, torch, tensorflow as tf
from collections import OrderedDict, defaultdict
from pathlib import Path
from tqdm import tqdm
from scipy.stats import ttest_rel

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
with open('benchmarking_results_23_01_2025-01_05.pkl', 'rb') as res_file:
    dice_x_benchmarking_results = pickle.load(res_file)

In [ ]:
backends = ['sklearn', 'PYT', 'TF2']
def load_torch_model(model_path, in_features):
    dummy_state_dict = torch.load(model_path)
    dummy_state_dict = {f'model.{key}': value for key, value in dummy_state_dict.items()}
    model = neuralnetworks.PYTModel(in_features)
    model.load_state_dict(dummy_state_dict)
    return model

def load_tensorflow_model(model_path):
    model = neuralnetworks.TF2Model()
    model.load_weights(model_path)
    return model

dataset_names = [
    "compas-recidivism",
    "adult-income",
    "lending-club",
    "german-credit"
]

dice_x_models = {}
for name in dataset_names:
    dice_x_models[name] = {}
    for backend in backends:
        if backend == 'sklearn':
            dice_x_models[name][backend] = dice_x_benchmarking_results[name][backend]['model']
        elif backend == 'PYT':
            model_path = dice_x_benchmarking_results[name][backend]['model_path']
            num_features = dice_x_benchmarking_results[name][backend]['metrics']['num_features']
            dice_x_models[name][backend] = load_torch_model(model_path, num_features)
        elif backend == 'TF2':
            model_path = dice_x_benchmarking_results[name][backend]['model_path']
            dice_x_models[name][backend] = load_tensorflow_model(model_path)

In [ ]:
# metric functions
def compute_validity(exp) -> float:
    return exp.get_validity_percentage()

def compute_mad(data_class: dice_ml.Data, normalized=False) -> dict:
    return data_class.get_valid_mads(normalized=normalized)

def compute_continuous_proximity(C: pd.DataFrame, x: pd.DataFrame, data_class: dice_ml.Data) -> float:
    mads = compute_mad(data_class)
    total_proximity = 0.0

    for feature in data_class.continuous_feature_names:
        diff = np.abs(C[feature] - x[feature].iloc[0])
        total_proximity += diff / mads[feature]

    return -np.mean(total_proximity)

def compute_continuous_proximity(C: pd.DataFrame, x: pd.DataFrame, data_class: dice_ml.Data) -> float:
    mads = compute_mad(data_class)
    normalized_diff = (np.abs(C[data_class.continuous_feature_names] - x[data_class.continuous_feature_names].iloc[0]) / 
                      np.array([mads[feature] for feature in data_class.continuous_feature_names]).reshape(1, -1))
    mean_per_CF = np.nanmean(normalized_diff, axis=1)
    return -np.mean(mean_per_CF)

def compute_categorical_proximity(C: pd.DataFrame, x: pd.DataFrame, data_class: dice_ml.Data) -> float:
    categorical_feats = data_class.categorical_feature_names
    if len(x) == 0 or len(categorical_feats) == 0:
        return 0.0
    x_values: pd.Series = x.iloc[0]
    diff_matrix: pd.DataFrame = C[categorical_feats] != x_values[categorical_feats]
    diff_count: pd.Series = diff_matrix.sum(axis=1)
    d_cat = len(categorical_feats)

    average_distance = diff_count.mean() / (d_cat * len(C)) if d_cat > 0 else 0.0

    return 1 - average_distance

def compute_continuous_diversity(C: pd.DataFrame, data_class: dice_ml.Data) -> float:
    cont_feats = data_class.continuous_feature_names

    if isinstance(C, pd.DataFrame):
        X = C[cont_feats].values
    else:
        X = C

    k, d = X.shape

    if k < 2 or d == 0:
        return 0.0
    
    mad_dict = compute_mad(data_class)
    mad_vector = np.array([
        mad_dict[feature] if mad_dict[feature] != 0 else 1.0
        for feature in cont_feats
    ]).reshape(1, d)
    
    diff = np.abs(X[:, None, :] - X[None, :, :])
    normalized_diff = diff / mad_vector

    pairwise_dist = np.mean(normalized_diff, axis=2)
    triu_indices = np.triu_indices(k, k=1)
    if len(triu_indices[0]) == 0:
        return 0.0
    average_distance = np.mean(pairwise_dist[triu_indices])
    return average_distance

def compute_continuous_diversity(C: pd.DataFrame, data_class: dice_ml.Data) -> float:
    mads = compute_mad(data_class)  
    X = C[data_class.continuous_feature_names].values
    diff = np.abs(X[:, None, :] - X[None, :, :])
    normalized_diff = diff / np.array([mads[feature] for feature in data_class.continuous_feature_names]).reshape(1, 1, -1)
    pairwise_dist = np.nanmean(normalized_diff, axis=2)
    triu_indices = np.triu_indices(len(C), k=1)
    return np.mean(pairwise_dist[triu_indices]) if len(triu_indices[0]) > 0 else 0.0

def compute_categorical_diversity(C: pd.DataFrame, data_class: dice_ml.Data) -> float:
    cat_feats = data_class.categorical_feature_names

    if isinstance(C, pd.DataFrame):
        X = C[cat_feats].values
    else:
        X = C
    
    k, d = X.shape

    if k < 2 or d == 0:
        return 0.0

    diff = (X[:, None, :] != X[None, :, :]).astype(np.float32)

    pairwise_dist = np.mean(diff, axis=2)
    triu_indices = np.triu_indices(k, k=1)
    if len(triu_indices[0]) == 0:
        return 0.0
    average_distance = np.mean(pairwise_dist[triu_indices])
    return average_distance

def compute_count_diversity(C: pd.DataFrame):
    if isinstance(C, pd.DataFrame):
        X = C.values
    else:
        X = C

    k, d = X.shape

    if X.size == 0 or k < 2 or d == 0:
        return 0.0
    
    diff = (X[:, None, :] != X[None, :, :]).astype(np.float32)
    triu_indices = np.triu_indices(k, k=1)
    total_diff = np.sum(diff[triu_indices[0], triu_indices[1], :])

    n_pairs = len(triu_indices[0])
    return total_diff / (n_pairs * d)

def compute_sparsity(C: pd.DataFrame, x: pd.DataFrame, data_class: dice_ml.Data) -> float:
    cont_feats = data_class.continuous_feature_names
    cont_CFs = C[cont_feats].to_numpy()
    cont_X = x[cont_feats].to_numpy()

    k, d = cont_CFs.shape

    diff = (cont_CFs != cont_X[0])

    num_changed = diff.sum()

    return 1 - (num_changed / (k * d))


def robustness_flip_rate(
    C: pd.DataFrame,
    target_col: str,
    data_iface: "dice_ml.Data",
    backend: str,
    model,
    *,
    noise_sd: float = 0.10,        # 10 % of feature range, per feature
    cat_flip_p: float = 0.20,      # 20 % chance to flip a categorical value
    n_repeat: int = 50,            # Monte-Carlo samples per CF
    rng: Optional[np.random.Generator] = None,
) -> float:
    """
    Robustness =  (#perturbed samples that KEEP the original class)
/ (total #perturbed samples)

    ↪ 1 = fully stable 0 = always flips
    """
    rng = rng or np.random.default_rng()

    # ------------------------------------------------------------------ helpers
    def _predict_class(X_raw: pd.DataFrame) -> np.ndarray:
        """Return class label 0/1 for X_raw (unencoded)."""
        if backend == "sklearn":
            return model.predict(X_raw)

        X_enc = data_iface.get_ohe_min_max_normalized_data(X_raw).values
        if backend == "PYT":
            import torch
            logits = model(torch.tensor(X_enc, dtype=torch.float32)
                          ).detach().cpu().numpy().ravel()
        elif backend == "TF2":
            import tensorflow as tf
            logits = model.predict(tf.constant(X_enc, dtype=tf.float32)).ravel()
        else:
            raise ValueError(f"Unknown backend {backend}")
        return (logits >= 0.5).astype(int)

    # ------------------------------------------------------------------ original predictions
    X_raw = C.drop(columns=[target_col]).reset_index(drop=True)
    y_orig = _predict_class(X_raw)

    # continuous & categorical bookkeeping
    cont_cols = data_iface.continuous_feature_names
    cat_cols  = data_iface.categorical_feature_names
    ranges    = data_iface.get_features_range_float()[1]          # {col: (lo, hi)}
    cat_vals  = {c: data_iface.get_features_range()[1][c] for c in cat_cols}

    n_cf   = len(C)
    n_mc   = n_repeat
    n_tot  = n_cf * n_mc
    kept   = 0

    # ------------------------------------------------------------------ Monte-Carlo loop
    for _ in range(n_repeat):
        X_noisy      = X_raw.copy()

        # ----- continuous perturbations
        for col in cont_cols:
            lo, hi = ranges[col]
            span   = hi - lo
            X_noisy[col] = np.clip(
                X_noisy[col] + rng.normal(0, noise_sd * span, size=n_cf),
                lo, hi
            )

        # ----- categorical flips (convert to object to avoid pandas category error)
        for col in cat_cols:
            X_noisy[col] = X_noisy[col].astype(object)
            mask = rng.random(n_cf) < cat_flip_p
            if mask.any():
                X_noisy.loc[mask, col] = rng.choice(cat_vals[col], size=mask.sum())

        # ----- predict & compare
        y_noisy = _predict_class(X_noisy)
        kept   += np.sum(y_noisy == y_orig)

    robustness = kept / n_tot
    return float(robustness)






In [ ]:
DATASETS   = [
    (helpers.load_compas_dataset(),  "twoyearrecid", "compas-recidivism"),
    (helpers.load_adult_income_dataset(), "income",  "adult-income"),
    (helpers.load_lending_club_dataset(), "loan_status", "lending-club"),
    (helpers.load_german_credit_dataset(), "credit_risk", "german-credit")
]
BACKENDS   = ["sklearn", "PYT", "TF2"]